In [ ]:
import json
import matplotlib.pyplot as plt

In [ ]:
train_filtered_users = json.load(open("../../data/processed/next-like-prediction-train-filtered.json"))
test_filtered_users = json.load(open("../../data/processed/next-like-prediction-test-filtered.json"))

In [ ]:
# user count = 
print(f"Train dataset user count: {len(train_filtered_users)}")
print(f"Test dataset user count: {len(test_filtered_users)}")

In [ ]:
# print predictions count
print(f"Train dataset predictions count: {sum(len(user['output']) for user in train_filtered_users)}")
print(f"Test dataset predictions count: {sum(len(user['output']) for user in test_filtered_users)}")

In [ ]:
# unique comments in in and output
unique_train_comments = set()
unique_test_comments = set()
unique_comments = set()
for user in train_filtered_users:
    for comment in user['input']:
        unique_comments.add(comment["text"])
        unique_train_comments.add(comment["text"])
    for comment in user['output']:
        unique_comments.add(comment["text"])
        unique_train_comments.add(comment["text"])

for user in test_filtered_users:
    for comment in user['input']:
        unique_comments.add(comment["text"])
        unique_test_comments.add(comment["text"])
    for comment in user['output']:
        unique_comments.add(comment["text"])
        unique_test_comments.add(comment["text"])

print(f"Unique comments count: {len(unique_comments)}")

# identify comments that are in both train and test and those that are only in train or only in test
train_only_comments = unique_train_comments - unique_test_comments
test_only_comments = unique_test_comments - unique_train_comments

print(f"Train only comments count: {len(train_only_comments)}")
print(f"Test only comments count: {len(test_only_comments)}")

In [ ]:
# Like Ratio in train and test datasets
for user in train_filtered_users:
    user["shared_input_and_output"] = user["input"] + user["output"]

for user in test_filtered_users:
    user["shared_input_and_output"] = user["input"] + user["output"]

# Calculate the like ratio for independent of the user
train_interactions = []
for user in train_filtered_users:
    for comment in user["shared_input_and_output"]:
        train_interactions.append(comment["label"])

test_interactions = []
for user in test_filtered_users:
    for comment in user["shared_input_and_output"]:
        test_interactions.append(comment["label"])

print(f"Train dataset like ratio: {sum(train_interactions) / len(train_interactions)}")
print(f"Test dataset like ratio: {sum(test_interactions) / len(test_interactions)}")

In [ ]:
train_like_ratios = [sum(comment["label"] for comment in user["shared_input_and_output"]) / len(user["shared_input_and_output"]) for user in train_filtered_users]
test_like_ratios = [sum(comment["label"] for comment in user["shared_input_and_output"]) / len(user["shared_input_and_output"]) for user in test_filtered_users]

print(f"Train dataset average like ratio: {sum(train_like_ratios) / len(train_like_ratios)}")
print(f"Test dataset average like ratio: {sum(test_like_ratios) / len(test_like_ratios)}")

# Plot the distribution of like ratios for train and test datasets

plt.hist(train_like_ratios, alpha=0.5, label="Train")
plt.hist(test_like_ratios, alpha=0.5, label="Test")
plt.xlabel("Like Ratio")
plt.ylabel("Count")
plt.title("Distribution of Like Ratios in Train and Test Datasets")
plt.legend()

In [ ]:
# output length distribution for train and test datasets
train_output_lengths = [len(user["output"]) for user in train_filtered_users]
test_output_lengths = [len(user["output"]) for user in test_filtered_users]

print(f"Train dataset average output length: {sum(train_output_lengths) / len(train_output_lengths)}")
print(f"Test dataset average output length: {sum(test_output_lengths) / len(test_output_lengths)}")

# Plot the distribution of output lengths for train and test datasets
plt.hist(train_output_lengths, alpha=0.5, label="Train")
plt.hist(test_output_lengths, alpha=0.5, label="Test")
plt.xlabel("Output Length")
plt.ylabel("Count")
plt.title("Distribution of Output Lengths in Train and Test Datasets")

In [ ]:
# generate median and mean of output lengths for train and test datasets
import numpy as np
print(f"Train dataset median output length: {np.median(train_output_lengths)}")
print(f"Test dataset median output length: {np.median(test_output_lengths)}")

In [ ]:
# average comment length in words for train and test datasets
train_comment_lengths = [len(comment["text"].split()) for user in train_filtered_users for comment in user["shared_input_and_output"]]
test_comment_lengths = [len(comment["text"].split()) for user in test_filtered_users for comment in user["shared_input_and_output"]]

print(f"Train dataset average comment length: {sum(train_comment_lengths) / len(train_comment_lengths)}")
print(f"Test dataset average comment length: {sum(test_comment_lengths) / len(test_comment_lengths)}")

In [ ]:
language_dict_train = {}
for user in train_filtered_users:
    for comment in user["shared_input_and_output"]:
        language = comment["language"]
        if language not in language_dict_train:
            language_dict_train[language] = 0
        language_dict_train[language] += 1

language_dict_test = {}
for user in test_filtered_users:
    for comment in user["shared_input_and_output"]:
        language = comment["language"]
        if language not in language_dict_test:
            language_dict_test[language] = 0
        language_dict_test[language] += 1

# print percentwise share of languages in train and test datasets
train_total = sum(language_dict_train.values())
test_total = sum(language_dict_test.values())

for language, count in language_dict_train.items():
    percent = (count / train_total) * 100
    print(f"Train dataset language: {language}, count: {count}, percent: {percent:.2f}%")

for language, count in language_dict_test.items():
    percent = (count / test_total) * 100
    print(f"Test dataset language: {language}, count: {count}, percent: {percent:.2f}%")